# 5. Combining grids, coverage, and densifying

This notebook builds new grids from installed ones:
1. **coverage:** which (Teff, log g, [M/H]) nodes a grid fills;
2. **combine:** several grids into one, here cool Kurucz stars with hot Husfeld stars;
3. **densify:** extra Teff nodes in a coarse cube, so interpolation between distant nodes does not step.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from sed_tools.api import SED
from sed_tools.cube.densify import densify_grid

plt.rcParams["figure.dpi"] = 80
SIGMA_SB = 5.670374419e-5
KURUCZ_CUT = dict(teff_min=4000, teff_max=8000, logg_min=3.0, logg_max=5.0,
                  metallicity_min=-1.0, metallicity_max=0.5)
# Notebook 1 downloads these grids; they are fetched here only if missing, so this notebook needs no network.
for name, cut in (("Kurucz2003all", KURUCZ_CUT), ("Husfeld", {}),):
    if not (SED._model_root / name / "lookup_table.csv").is_file():
        SED.fetch(name, **cut)
stellar_models = SED._model_root

KURUCZ = "Kurucz2003all/Kurucz2003all_alpha_0p00_lh_1p25_vtur_2_km_s"
HUSFELD = "Husfeld/Husfeld_yhe_0p10"

## 1. Coverage

`SED.coverage(name)` prints each axis's range, values and spacing, and the fill fraction: the share of the Teff × log g × [M/H] product that holds a spectrum. Interpolation next to an empty node is pulled toward zero, so the fill fraction matters.
`filters=` also lists bands the grid's wavelengths do not sample.

In [ ]:
report = SED.coverage(KURUCZ, plot=False, filters=["2MASS/2MASS"])
print({key: report[key] for key in ("n_spectra", "n_distinct_nodes", "fill_fraction")})

## 2. Combining grids

`SED.combine([...], output=name)` places every spectrum of the given grids on one set of axes (the union of their Teff, log g and [M/H] values) and one wavelength axis (the union of theirs). Each spectrum keeps its own flux; a node no spectrum fills stays at zero.
Grids are named by their folder under the stellar model directory, so one variant of a split grid is named `grid/variant`. A combination whose extra axes vary is split into variants, as one grid is.

Husfeld gives no [M/H], so its spectra fill every [M/H] plane of the combination.

In [ ]:
combined = SED.combine([KURUCZ, HUSFELD], output="Kurucz_Husfeld", visualization=False)
print(combined.parameter_ranges())

In [ ]:
report = SED.coverage("Kurucz_Husfeld", plot=False)
print({key: report[key] for key in ("n_spectra", "n_distinct_nodes", "fill_fraction")})

The combination covers 4000–8000 K from Kurucz and 35000–80000 K from Husfeld. Between 8000 and 35000 K, and at log g values only one grid has, it has no spectra: interpolating there blends real spectra with empty (zero) nodes. Check coverage before using a combination.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
for teff, logg in ((6000, 4.5), (40000, 5.0)):
    s = combined(teff, logg, 0.0)
    ratio = np.trapezoid(s.flux, s.wavelength) / (SIGMA_SB * teff**4)
    print(f"{teff} K, log g {logg}: ∫F/σT⁴ = {ratio:.3f}")
    assert 0.9 < ratio < 1.1
    ax.loglog(s.wavelength, s.flux, lw=0.7, label=f"{teff} K, log g {logg}")
ax.set(xlim=(100, 1e5), ylim=(1e2, 1e12), xlabel="Wavelength (Å)", ylabel="F$_\\lambda$")
ax.legend();

## 3. Densifying a coarse cube

Husfeld's Teff values are 5000 K apart. `densify_grid(source_cube, output_cube, teff_spacing=...)` writes a cube with extra Teff nodes. A new node is a Planck spectrum scaled to the nearest real spectrum's bolometric flux, or, with `ml_model=`, a trained ML generator's spectrum where the point lies inside its training range (notebook 7). The source's `lookup_table.csv` is copied beside the output for MESA.

The source is a 3-axis cube: one variant's `flux_cube.bin`.

In [ ]:
source = stellar_models / HUSFELD / "flux_cube.bin"
dense_cube = stellar_models / "Husfeld_yhe_0p10_dense" / "flux_cube.bin"
densify_grid(str(source), str(dense_cube), teff_spacing=2500)

In [ ]:
coarse = SED.local(HUSFELD)
dense = SED.local("Husfeld_yhe_0p10_dense")
print("Teff nodes:", coarse.model().teff_grid, "->", dense.model().teff_grid)

fig, ax = plt.subplots(figsize=(7, 3.5))
teffs = np.arange(35000, 80001, 500)
for model, label in ((coarse, "Husfeld yhe 0.1"), (dense, "densified, 2500 K")):
    ratios = []
    for teff in teffs:
        s = model(teff, 5.0, 0.0)
        ratios.append(np.trapezoid(s.flux, s.wavelength) / (SIGMA_SB * teff**4))
    ax.plot(teffs, ratios, label=label)
ax.set(xlabel="Teff (K)", ylabel="∫F dλ / σT⁴", title="log g 5.0")
ax.legend();

## Command-line equivalents

| Python | Command line |
|---|---|
| `SED.coverage(name)` | `sed-tools coverage --grids NAME [--no-plot]` |
| `SED.combine([A, B], output=C)` | `sed-tools combine --grids A B --output C` |
| `densify_grid(src, dst, teff_spacing=2500)` | `sed-tools densify --flux-cube SRC --output DST --teff-spacing 2500 [--ml-model DIR]` |

In [ ]:
!sed-tools coverage --grids Kurucz_Husfeld --no-plot